In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import datetime
from scipy.ndimage import gaussian_filter1d
from scipy import stats

# Paths to your data files
VRlog_path = r"D:\V1_SpatialModulation\V1_SpatialMod_VRLog\VRlog_JSY038_05012025_02-18-59.txt"
treadmill_file_path = r"D:\V1_SpatialModulation\V1_SpatialMod_VRLog\TMlog_05012025_02-18-59.txt"

# Initialize lists to store data
treadmill_abs_times = []
treadmill_distances = []
treadmill_speeds = []

# Read the file
with open(treadmill_file_path, 'r') as file:
    lines = file.readlines()
    
    # Skip header lines
    data_start = 0
    for i, line in enumerate(lines):
        if "Log format is current time distance speed" in line:
            data_start = i + 1
            break
    
    # Process each data line
    for line in lines[data_start:]:
        parts = line.strip().split()
        if len(parts) >= 3:  # Ensure we have time, distance, and speed
            try:
                time_str = parts[0]
                distance = float(parts[1])
                speed = float(parts[2])
                
                # Convert time string to datetime
                time_obj = datetime.datetime.strptime(time_str, '%H.%M.%S.%f')
                
                treadmill_abs_times.append(time_obj)
                treadmill_distances.append(distance)
                treadmill_speeds.append(speed)
            except (ValueError, IndexError):
                continue  # Skip malformed lines

# Convert to numpy arrays
treadmill_abs_times = np.array(treadmill_abs_times)
treadmill_distances = np.array(treadmill_distances)
treadmill_speeds = np.array(treadmill_speeds)

treadmill_distances = treadmill_distances - treadmill_distances[0]  # Normalize distances to start at 0
treadmill_speeds[treadmill_speeds > 500] = 500

# Calculate relative times (seconds from start)
if len(treadmill_abs_times) > 0:
    start_time = treadmill_abs_times[0]
    treadmill_rel_times = np.array([(t - start_time).total_seconds() for t in treadmill_abs_times])
else:
    treadmill_rel_times = np.array([])

print(f"Loaded {len(treadmill_abs_times)} treadmill data points")


# Read the VR log file
rawVR_data = []
with open(VRlog_path, "r") as file:
    lines = file.readlines()
    # Skip header lines (first 3 lines typically)
    for line in lines[3:]:
        rawVR_data.append(line.strip().split("\t"))

# Extract data - handle different formats for safety
vr_abs_times = []
vr_rel_times = []
vr_events = []
vr_locations = []

for line in rawVR_data:
    if len(line) >= 4:  # Basic sanity check
        vr_abs_times.append(line[0])
        vr_rel_times.append(float(line[1]) if line[1].replace('.','',1).isdigit() else 0)
        vr_events.append(line[2])
        
        # Check which column has location data based on event type
        if line[2] == 'p' and len(line) >= 6:
            vr_locations.append(float(line[5]) if line[5].replace('.','',1).replace('-','',1).isdigit() else 0)
        else:
            vr_locations.append(float(line[3]) if line[3].replace('.','',1).replace('-','',1).isdigit() else 0)

# Convert to numpy arrays
vr_abs_times = np.array(vr_abs_times)
vr_rel_times = np.array(vr_rel_times)
vr_events = np.array(vr_events)
vr_locations = np.array(vr_locations)

# Find the first 's' event (start signal)
s_indices = np.where(vr_events == 's')[0]
if len(s_indices) > 0:
    start_index = s_indices[0]
    # Truncate arrays to start at the 's' event
    vr_abs_times = vr_abs_times[start_index:]
    vr_rel_times = vr_rel_times[start_index:]
    vr_events = vr_events[start_index:]
    vr_locations = vr_locations[start_index:]

# # Parse absolute times
# abs_times = []
# for t_str in vr_absoluteT:
#     try:
#         time_obj = datetime.datetime.strptime(t_str, '%H.%M.%S.%f')
#         abs_times.append(time_obj)
#     except ValueError:
#         # Handle parsing errors
#         if abs_times:
#             abs_times.append(abs_times[-1])
#         else:
#             abs_times.append(datetime.datetime.now())  # Placeholder

# abs_times = np.array(abs_times)

# # Calculate relative times
# rel_times = np.array([0.0] * len(abs_times))
# if len(abs_times) > 0:
#     start_time = abs_times[0]
#     rel_times = np.array([(t - start_time).total_seconds() for t in abs_times])

print(f"Loaded {len(vr_abs_times)} VR data points")

In [ ]:
# Calculate time offset between the two datasets
treadmill_start = treadmill_abs_times[0]
vr_start = datetime.datetime.strptime(vr_abs_times[0], '%H.%M.%S.%f')  # Convert string to datetime
time_offset = (treadmill_start - vr_start).total_seconds()

print(f"Time offset between datasets: {time_offset:.3f} seconds")
print(f"Treadmill data starts at: {treadmill_start}")
print(f"VR data starts at: {vr_start}")

# find index of treadmill_rel_times that is closest to time_offset
closest_index = np.argmin(np.abs(treadmill_rel_times - time_offset))
print(f"Closest index in treadmill data to time offset: {closest_index}")
print(f"Closest treadmill relative time to offset: {treadmill_rel_times[closest_index]}")

treadmill_abs_times = treadmill_abs_times[closest_index:]  # Truncate treadmill data to start at the same time as VR data
treadmill_rel_times = treadmill_rel_times[closest_index:]  # Truncate treadmill relative times
treadmill_distances = treadmill_distances[closest_index:]  # Truncate distances
treadmill_speeds = treadmill_speeds[closest_index:]  # Truncate speeds

# Constants for VR-to-cm conversion (from spatial discretization)
single_revolution_VR = 282.415
single_revolution_treadmill = 27.8
single_lap_VR = 1320.645683  # Adjust based on your specific VR environment
single_lap_treadmill = single_revolution_treadmill * single_lap_VR / single_revolution_VR

print(f"Converting VR locations to cm using single_lap_treadmill = {single_lap_treadmill:.2f} cm")

# Convert VR locations to cm
raw_locations = vr_locations


# Find the range of VR locations
min_loc = np.min(raw_locations)
max_loc = np.max(raw_locations)
location_range = max_loc - min_loc

# Scale and shift to match real-world distances
locations_cm = (raw_locations - min_loc) / location_range * single_lap_treadmill
print(f"Original VR location range: {min_loc:.2f} to {max_loc:.2f}")
print(f"Converted position range: 0.00 to {single_lap_treadmill:.2f} cm")

# Interpolate treadmill data (distance, speed) to VR time points
interp_distances = np.interp(
    vr_rel_times,
    treadmill_rel_times,
    treadmill_distances,
    left=np.nan,
    right=np.nan
)

interp_speeds = np.interp(
    vr_rel_times,
    treadmill_rel_times,
    treadmill_speeds,
    left=np.nan,
    right=np.nan
)

plt.plot(vr_rel_times, interp_distances, label='Treadmill Distance (cm)', color='blue', alpha=0.5)
# plt.plot(vr_rel_times, interp_speeds, label='Treadmill Speed (cm/s)', color='red', alpha=0.5)


# Combine the datasets
aligned_data = {
    'times': vr_rel_times,
    'vr_locations': locations_cm,  # Converted to cm
    'vr_events': vr_events,
    'raw_locations': raw_locations,  # Keep original for reference
    'treadmill_distances': interp_distances,
    'treadmill_speeds': interp_speeds,
    'time_offset': time_offset,
    'conversion_factor': single_lap_treadmill / location_range
}


In [ ]:
print(n_laps)

In [ ]:
def reshape_into_laps(location, speed, high_percentile=90, low_percentile=10, plot_detection=True):
    """
    Reshape both spike data and location data into laps using percentile-based 
    thresholds for more robust lap detection. Preserves all data points in each lap.
    
    Parameters:
    -----------
    spks : numpy.ndarray
        Spike data (cells x time)
    location : numpy.ndarray
        Location data (time,)
    high_percentile : float, optional
        Percentile value to use for high threshold (default: 90)
    low_percentile : float, optional
        Percentile value to use for low threshold (default: 10)
    plot_detection : bool, optional
        Whether to plot the lap detection result (default: True)
        
    Returns:
    --------
    spks_laps : list
        List of numpy arrays containing spike data for each lap (each array is cells x lap_length)
    location_laps : list
        List of numpy arrays containing location data for each lap (each array is lap_length,)
    n_laps : int
        Number of laps detected
    """
    # Find thresholds based on percentiles instead of absolute values
    threshold_high = np.percentile(location, high_percentile)
    threshold_low = np.percentile(location, low_percentile)
    
    # print(f"Location range: {np.min(location):.2f} to {np.max(location):.2f}")
    # print(f"Using high threshold: {threshold_high:.2f} (at {high_percentile}th percentile)")
    # print(f"Using low threshold: {threshold_low:.2f} (at {low_percentile}th percentile)")
    
    # Find the first minimum in the data
    first_min_indices = np.where(location < threshold_low)[0]
    if len(first_min_indices) == 0:
        print("No location values below low threshold found!")
        return None, None, 0
    first_min_idx = first_min_indices[0]
    
    # Use a state machine approach to identify transitions
    lap_ends = []
    state = "low" if location[0] < threshold_low else "high"
    
    for i in range(1, len(location)):
        if state == "high" and location[i] < threshold_low:
            lap_ends.append(i)
            state = "low"
        elif state == "low" and location[i] > threshold_high:
            state = "high"
    
    lap_ends = np.array(lap_ends)
    
    if len(lap_ends) == 0:
        print("No lap transitions detected!")
        return None, None, 0
        
    # Add start and end indices
    lap_starts = np.concatenate(([first_min_idx], lap_ends))
    lap_ends = np.concatenate((lap_ends, [len(location)]))
    
    # Verify lap detection - ensure we have clear transitions
    valid_laps = []
    valid_starts = []
    valid_ends = []
    
    for i, (start, end) in enumerate(zip(lap_starts, lap_ends)):
        lap_loc = location[start:end]
        # Check if this lap has a clear high point
        if np.max(lap_loc) > threshold_high:
            valid_laps.append(i)
            valid_starts.append(start)
            valid_ends.append(end)
    
    if len(valid_laps) < len(lap_starts):
        print(f"Removed {len(lap_starts) - len(valid_laps)} incomplete laps")
        lap_starts = np.array(valid_starts)
        lap_ends = np.array(valid_ends)
    
    # Calculate lap lengths
    lap_lengths = lap_ends - lap_starts
    n_laps = len(lap_starts)
    
    # print(f"Found {n_laps} laps")
    # print(f"Lap lengths: {lap_lengths}")
    
    # Plot lap detection for verification
    if plot_detection:
        plt.figure(figsize=(15, 5))
        plt.plot(location, 'b-', alpha=0.5)
        
        # Plot lap transitions
        # For lap ends, use all except the last element (which is the end of data)
        if len(lap_ends) > 1:  # Make sure we have actual lap ends to plot
            plt.plot(lap_ends[:-1], location[lap_ends[:-1]], 'r.', markersize=10, label='Lap Ends')
        
        # For lap starts, use all elements
        plt.plot(lap_starts, location[lap_starts], 'g.', markersize=10, label='Lap Starts')
        
        # Add threshold lines
        plt.axhline(y=threshold_high, color='gray', linestyle='--', alpha=0.5, label='High Threshold')
        plt.axhline(y=threshold_low, color='gray', linestyle='--', alpha=0.5, label='Low Threshold')
        
        plt.title(f'Lap Detection (Found {n_laps} laps)')
        plt.xlabel('Time')
        plt.ylabel('Location')
        plt.legend()
        # plt.show()

    # Create lists to store data for each lap
    location_laps = []
    speed_laps = []
    
    # Fill in the data with full lap lengths
    for start, end in zip(lap_starts, lap_ends):
        location_laps.append(location[start:end])
        speed_laps.append(speed[start:end])
    
    return location_laps, speed_laps, n_laps

location_laps, speed_laps, n_laps = reshape_into_laps(aligned_data['vr_locations'], aligned_data['treadmill_speeds'], plot_detection=True)
filtered_speed_laps = []
filtered_location_laps = []

for i in range(n_laps):
    lap_speed = speed_laps[i]
    lap_loc = location_laps[i]
    
    # Detect inactivity within this lap
    location_diff = np.abs(np.diff(lap_loc, n=5))
    inactivity_threshold=1e-2
    frames_to_keep = 0
    stationary_mask = location_diff < inactivity_threshold
    stationary_indices = np.where(stationary_mask)[0]
    
    # Group consecutive stationary indices
    from itertools import groupby
    from operator import itemgetter
    
    def group_consecutive(data):
        for k, g in groupby(enumerate(data), lambda x: x[0] - x[1]):
            yield list(map(itemgetter(1), g))
    
    stationary_periods = list(group_consecutive(stationary_indices))
    
    # Create mask for which frames to keep
    mask = np.ones(len(lap_loc), dtype=bool)
    
    for period in stationary_periods:
        if len(period) > 2 * frames_to_keep:
            start_idx = period[0]
            end_idx = period[-1]
            
            # Keep beginning and end, remove middle
            mask[start_idx + frames_to_keep:end_idx - frames_to_keep + 1] = False
    
    # Apply mask
    filtered_speed_laps.append(lap_speed[mask])
    filtered_location_laps.append(lap_loc[mask])

# for lap_idx in range(n_laps):
#     original_frames = len(location_laps[lap_idx])
#     filtered_frames = len(filtered_location_laps[lap_idx])
#     print(f"  Lap {lap_idx+1}: {filtered_frames}/{original_frames} frames kept" + 
#             f" ({filtered_frames/original_frames*100:.1f}%)")


In [ ]:
print(f"original length of location: {aligned_data['vr_locations'].shape[0]}")
print(f"number of laps: {n_laps}")

print(f"length of location_laps: {len(speed_laps[10])}")
print(f"length of location_laps: {len(location_laps[10])}")

print(f"length of filtered location_laps: {len(filtered_speed_laps[10])}")
print(f"length of filtered location_laps: {len(filtered_location_laps[10])}")

In [ ]:
# Define number of bins
n_bins = round(single_lap_treadmill)
print(f"Using {n_bins} spatial bins")

# Create spatial bins based on the range of all location data
location_min = min(np.min(loc) for loc in location_laps)
location_max = max(np.max(loc) for loc in location_laps)
spatial_bins = np.linspace(location_min, location_max, n_bins + 1)
bin_centers = (spatial_bins[:-1] + spatial_bins[1:]) / 2

# Initialize arrays
speed_activity = np.zeros((n_laps, n_bins))
trial_averaged_speed_activity = np.zeros((n_bins))

# Process each lap
for lap_idx, (lap_spks, lap_location) in enumerate(zip(speed_laps, location_laps)):
    # Assign timepoints to spatial bins
    bin_indices = np.clip(np.digitize(lap_location, spatial_bins) - 1, 0, n_bins-1)
    
    # Calculate activity for each bin in this lap
    for bin_idx in range(n_bins):
        bin_mask = (bin_indices == bin_idx)
        if np.any(bin_mask):
            speed_activity[lap_idx, bin_idx] = np.sum(lap_spks[bin_mask])
            # Normalize by number of timepoints in this bin
            speed_activity[lap_idx, bin_idx] /= np.sum(bin_mask)

# Calculate trial average
trial_averaged_speed_activity = np.mean(speed_activity, axis=0)

print("Completed spatial discretization")

In [ ]:
print(round(n_laps/8))
a = [0, 100, 200, 300, 400, 500, 600, 700]
average_speedfor5laps = np.zeros((round(n_laps/5), n_bins))
for i in range(round(n_laps/8)):
    average_speedfor5laps[i] = np.mean(speed_activity[i*5:(i+1)*5], axis=0)
    print(f"average speed for lap {i}: {np.shape(average_speedfor5laps[i])}")
for i in range(round(n_laps/8)):
    plt.plot(bin_centers, average_speedfor5laps[i]+a[i], alpha=0.5)
